# Dimension scaling and λ_GL calibration

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Section 5.6, Appendix M, Table 15.

Measures raw and entry-normalized determining residuals on the ring-coupled double-well family for d ∈ {1, 3, 20, 50} and selects λ_GL on validation data. Run once per `STATE_DIM` (restarting the kernel between runs); the final cell aggregates across dimensions.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Config, imports, reproducibility, dimension-controlled knobs

In [ ]:
# ============================================================================
# Cell 1 — Config, imports, reproducibility, dimension-controlled knobs
# ============================================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import csv
import json
import math
import os
import time
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())

CONFIG = {
    # Reproducibility
    "SEED": 42,

    # ------------------------------------------------------------------
    # E8 dimension control
    # Run the notebook separately with STATE_DIM = 1, 3, 20, and 50.
    # ------------------------------------------------------------------
    "STATE_DIM": 1,
    "E8_DIMENSIONS": [1, 3, 20, 50],
    "E8_RUN_TAG": "dimension_scaling",
    "E8_RESULTS_DIR": "./e8_results",

    # One common scalable SDE family is used at every dimension so that
    # dimension is not confounded with a different dynamical system.
    "TRUE_SDE_NAME": "ring_double_well_scalable",

    # Synthetic trajectories
    "T_FINAL": 2.0,
    "DT": 0.01,
    "N_TRAJECTORIES": 64,
    "X0_MEAN": 0.0,
    "X0_STD": 0.75,
    "TRAIN_FRACTION": 0.70,
    "VALID_FRACTION": 0.15,

    # Neural SDE surrogate
    "MLP_HIDDEN_DIM": 192,
    "MLP_DEPTH": 3,
    "INPUT_T_SCALE": 2.0,
    "INPUT_X_SCALE": 2.5,
    "SIGMA_EPS": 1e-4,

    # Stage I
    "BASELINE_EPOCHS": 2000,
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2e-3,
    "PRINT_EVERY": 250,

    # Stage II
    # A common m=3 is used across dimensions to isolate dimension scaling.
    "N_GENERATORS": 3,
    "SYM_DISCOVERY_EPOCHS": 4000,
    "SYM_DISCOVERY_LR": 5e-3,
    "SYM_DISCOVERY_POINTS": 256,
    "SYM_INIT_SCALE": 0.03,
    "SYM_JITTER_STD": 0.05,

    # Stage-II loss weights
    "LAMBDA_DET": 1.0,
    "LAMBDA_CLOSE": 0.5,
    "LAMBDA_IND": 0.05,
    "LAMBDA_PROJECTABLE": 1.0,
    "LAMBDA_NORM_HINGE": 0.01,
    "MIN_GENERATOR_NORM": 0.10,
    "CLOSURE_RCOND": 1e-5,

    # Editable basis library
    "BASIS_LINEAR_MU_LIST": [-1.0, -0.5, 0.5, 1.0],
    "BASIS_EXP_CLIP": 6.0,
    "BASIS_LOW_RANK_K": 6,
    "BASIS_ADD_EXPONENTIAL": True,
    "BASIS_ADD_LOW_RANK_ROTATIONS": True,
    "BASIS_ADD_LOW_RANK_SHEARS": True,

    # Stage III common training budget
    "STAGE_F_EPOCHS": 800,
    "STAGE_F_BATCH_SIZE": 512,
    "STAGE_F_LR": 3e-4,
    "STAGE_F_N_DATA_SYM_POINTS": 128,
    "STAGE_F_N_EXTRA_SYM_POINTS": 128,
    "STAGE_F_JITTER_STD": 0.05,
    "STAGE_F_LAMBDA_PROJECTABLE": 1e-3,

    # Submitted reference weight. This uses exactly the same per-entry
    # determining loss as every other Stage-III branch; only lambda differs.
    "E8_RUN_LEGACY_BRANCH": True,
    "E8_LEGACY_LAMBDA": 105000.0,

    # Per-entry determining-loss weight sweep. This range spans the
    # approximate gradient-parity regime and approaches the submitted weight.
    "E8_NORMALIZED_LAMBDAS": [
        1.0e2,
        3.0e2,
        1.0e3,
        3.0e3,
        1.0e4,
        3.0e4,
        1.0e5,
    ],

    # Collocation distributions.
    # E8 isolates scaling, so the primary distribution is held fixed.
    "E8_EVAL_COLLOCATION_MODE": "data_jitter",
    "E8_VARIANCE_COLLOCATION_MODE": "data_jitter",

    # E8 fixed diagnostics
    "N_EVAL_POINTS": 2000,
    "E8_RESIDUAL_EVAL_POINTS": 512,
    "E8_GRAD_DATA_BATCH_SIZE": 256,
    "E8_GRAD_COLLOCATION_POINTS": 128,
    "E8_MC_BATCH_SIZES": [16, 32, 64, 128, 256, 512],
    "E8_MC_REPEATS": 20,
    "E8_MC_REFERENCE_BRANCH": "baseline",

    "E8_RESIDUAL_DIAGNOSTIC_SPLIT": "test",
    "E8_SELECTION_REQUIRE_BASELINE_IMPROVEMENT": True,
    "E8_SELECTION_MIN_VALID_NLL_GAIN": 0.0,
    "E8_RUN_RESIDUAL_CONSISTENCY_CHECK": True,
    "E8_RESIDUAL_CHECK_POINTS": 512,
    "E8_RESIDUAL_CHECK_RTOL": 1e-6,
    "E8_RESIDUAL_CHECK_ATOL": 1e-9,
}

D = int(CONFIG["STATE_DIM"])
Q = D + 1

if D not in CONFIG["E8_DIMENSIONS"]:
    raise ValueError(
        f"STATE_DIM={D} is not in E8_DIMENSIONS={CONFIG['E8_DIMENSIONS']}."
    )

if CONFIG["N_GENERATORS"] > D + 2:
    raise ValueError(
        "N_GENERATORS must not exceed the projectable-algebra upper-bound "
        f"search range d+2={D+2}."
    )

key = jax.random.PRNGKey(int(CONFIG["SEED"]))

print("\nE8 configuration")
print("  dimension       :", D)
print("  coordinates     :", Q)
print("  SDE family      :", CONFIG["TRUE_SDE_NAME"])
print("  generators      :", CONFIG["N_GENERATORS"])
print("  normalized λs   :", CONFIG["E8_NORMALIZED_LAMBDAS"])


## Cell 2 — Dimension-scalable SDE family and trajectory data

In [ ]:
# ============================================================================
# Cell 2 — Dimension-scalable SDE family and trajectory data
# ============================================================================

def _phase_vector():
    return 2.0 * jnp.pi * jnp.arange(D, dtype=jnp.float32) / float(max(D, 1))


def true_drift(t, x):
    """Dimension-scalable ring-coupled double-well drift."""
    coupling = 0.25
    lap = jnp.roll(x, 1, axis=-1) + jnp.roll(x, -1, axis=-1) - 2.0 * x
    forcing = 0.08 * jnp.sin(2.0 * jnp.pi * t + _phase_vector())
    return x - x**3 + coupling * lap + forcing


def true_sigma(t, x):
    """Positive state-dependent diagonal diffusion at every dimension."""
    phase = _phase_vector()
    base = 0.18 + 0.02 * jnp.sin(phase) ** 2
    return base + 0.04 * jax.nn.sigmoid(x)


ACTIVE_SDE = {
    "name": CONFIG["TRUE_SDE_NAME"],
    "description": (
        "Dimension-scalable ring-coupled double-well SDE with "
        "state-dependent diagonal diffusion."
    ),
}


def simulate_one_trajectory(key, x0):
    n_steps = int(round(CONFIG["T_FINAL"] / CONFIG["DT"]))
    dt = jnp.float32(CONFIG["DT"])
    noise_keys = jax.random.split(key, n_steps)
    step_ids = jnp.arange(n_steps, dtype=jnp.int32)

    def step(x, inputs):
        step_id, noise_key = inputs
        t = step_id.astype(jnp.float32) * dt
        eps = jax.random.normal(noise_key, (D,), dtype=jnp.float32)
        x_next = x + true_drift(t, x) * dt + true_sigma(t, x) * jnp.sqrt(dt) * eps
        return x_next, x_next

    _, xs_after = jax.lax.scan(step, x0, (step_ids, noise_keys))
    return jnp.concatenate([x0[None, :], xs_after], axis=0)


simulate_one_trajectory_jit = jax.jit(simulate_one_trajectory)


def build_transition_dataset(paths):
    n_traj, n_time, _ = paths.shape
    n_steps = n_time - 1

    t0 = jnp.tile(
        jnp.arange(n_steps, dtype=jnp.float32) * CONFIG["DT"],
        n_traj,
    )
    x0 = paths[:, :-1, :].reshape(-1, D)
    x1 = paths[:, 1:, :].reshape(-1, D)
    dt = jnp.full((n_traj * n_steps,), CONFIG["DT"], dtype=jnp.float32)
    dx = x1 - x0
    return t0, x0, dt, dx


def sample_batch(key, dataset, batch_size):
    n = dataset[0].shape[0]
    idx = jax.random.randint(key, (batch_size,), 0, n)
    return tuple(arr[idx] for arr in dataset)


# Simulate trajectories.
key, key_x0, key_paths = jax.random.split(key, 3)
x0_all = (
    CONFIG["X0_MEAN"]
    + CONFIG["X0_STD"] * jax.random.normal(key_x0, (CONFIG["N_TRAJECTORIES"], D))
)
trajectory_keys = jax.random.split(key_paths, CONFIG["N_TRAJECTORIES"])
paths = jax.vmap(simulate_one_trajectory_jit)(trajectory_keys, x0_all)

n_train = int(round(CONFIG["TRAIN_FRACTION"] * CONFIG["N_TRAJECTORIES"]))
n_valid = int(round(CONFIG["VALID_FRACTION"] * CONFIG["N_TRAJECTORIES"]))
n_test = CONFIG["N_TRAJECTORIES"] - n_train - n_valid

if min(n_train, n_valid, n_test) < 1:
    raise ValueError("Train/validation/test trajectory split produced an empty partition.")

train_paths = paths[:n_train]
valid_paths = paths[n_train:n_train + n_valid]
test_paths = paths[n_train + n_valid:]

train_data = build_transition_dataset(train_paths)
valid_data = build_transition_dataset(valid_paths)
test_data = build_transition_dataset(test_paths)

print("\nSynthetic data")
print("  paths shape       :", paths.shape)
print("  train trajectories:", n_train)
print("  valid trajectories:", n_valid)
print("  test trajectories :", n_test)
print("  train transitions :", train_data[0].shape[0])
print("  valid transitions :", valid_data[0].shape[0])
print("  test transitions  :", test_data[0].shape[0])


## Cell 3 — Dynamic neural SDE, optimizer, and Stage-I baseline

In [ ]:

# ============================================================================
# Cell 3 — Dynamic neural SDE, optimizer, and Stage-I baseline
# ============================================================================

def glorot_uniform(key, in_dim, out_dim):
    lim = jnp.sqrt(6.0 / float(in_dim + out_dim))
    return jax.random.uniform(key, (in_dim, out_dim), minval=-lim, maxval=lim)


class MLP:
    def __init__(self, in_dim, hidden_dim, depth, out_dim):
        self.in_dim = int(in_dim)
        self.hidden_dim = int(hidden_dim)
        self.depth = int(depth)
        self.out_dim = int(out_dim)

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)
        params = []
        for k, in_d, out_d in zip(keys, sizes[:-1], sizes[1:]):
            params.append({
                "W": glorot_uniform(k, in_d, out_d),
                "b": jnp.zeros((out_d,), dtype=jnp.float32),
            })
        return params

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        return h @ params[-1]["W"] + params[-1]["b"]


surrogate_model = MLP(
    in_dim=Q,
    hidden_dim=CONFIG["MLP_HIDDEN_DIM"],
    depth=CONFIG["MLP_DEPTH"],
    out_dim=2 * D,
)


def normalize_input(t, x):
    tn = jnp.asarray([t / CONFIG["INPUT_T_SCALE"]], dtype=jnp.float32)
    xn = x / CONFIG["INPUT_X_SCALE"]
    return jnp.concatenate([tn, xn], axis=0)


def surrogate_f_sigma(params, t, x):
    raw = surrogate_model.apply(params, normalize_input(t, x))
    f = raw[:D]
    sigma = jax.nn.softplus(raw[D:]) + CONFIG["SIGMA_EPS"]
    return f, sigma


def f_vec_single(params, z):
    return surrogate_f_sigma(params, z[0], z[1:])[0]


def sigma_vec_single(params, z):
    return surrogate_f_sigma(params, z[0], z[1:])[1]


def batched_surrogate_values(params, t_values, x_values):
    return jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(
        t_values, x_values
    )


def data_nll_loss(params, batch):
    t0, x0, dt, dx = batch
    f, sigma = batched_surrogate_values(params, t0, x0)
    mean = f * dt[:, None]
    var = sigma**2 * dt[:, None] + 1e-8
    nll_components = 0.5 * (
        (dx - mean) ** 2 / var + jnp.log(2.0 * jnp.pi * var)
    )
    return jnp.mean(jnp.sum(nll_components, axis=-1))


def adam_init(params):
    return {
        "m": jax.tree_util.tree_map(jnp.zeros_like, params),
        "v": jax.tree_util.tree_map(jnp.zeros_like, params),
        "t": jnp.array(0, dtype=jnp.int32),
    }


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    t_new = state["t"] + 1
    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state["m"], grads,
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * (g * g),
        state["v"], grads,
    )
    m_hat = jax.tree_util.tree_map(lambda m: m / (1.0 - beta1**t_new), m_new)
    v_hat = jax.tree_util.tree_map(lambda v: v / (1.0 - beta2**t_new), v_new)
    params_new = jax.tree_util.tree_map(
        lambda p, m, v: p - lr * m / (jnp.sqrt(v) + eps),
        params, m_hat, v_hat,
    )
    return params_new, {"m": m_new, "v": v_new, "t": t_new}


def tree_l2_norm(tree):
    leaves = jax.tree_util.tree_leaves(tree)
    return jnp.sqrt(sum(jnp.sum(x**2) for x in leaves) + 1e-30)


@jax.jit
def baseline_train_step(params, opt_state, batch):
    loss, grads = jax.value_and_grad(data_nll_loss)(params, batch)
    params, opt_state = adam_update(
        params, grads, opt_state, CONFIG["BASELINE_LR"]
    )
    return params, opt_state, loss


key, subkey = jax.random.split(key)
sde_params = surrogate_model.init(subkey)
sde_opt_state = adam_init(sde_params)
baseline_losses = []

start = time.time()
for epoch in range(1, CONFIG["BASELINE_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    batch = sample_batch(subkey, train_data, CONFIG["BASELINE_BATCH_SIZE"])
    sde_params, sde_opt_state, loss = baseline_train_step(
        sde_params, sde_opt_state, batch
    )
    baseline_losses.append(float(loss))

    if epoch == 1 or epoch % CONFIG["PRINT_EVERY"] == 0:
        print(
            f"[Stage I] epoch {epoch:5d} | "
            f"batch {float(loss): .6f} | "
            f"train {float(data_nll_loss(sde_params, train_data)): .6f} | "
            f"valid {float(data_nll_loss(sde_params, valid_data)): .6f} | "
            f"test {float(data_nll_loss(sde_params, test_data)): .6f}"
        )

baseline_runtime_sec = time.time() - start
sde_params_baseline = sde_params
print(f"Stage-I runtime: {baseline_runtime_sec:.2f} sec")

## Cell 4 — Dynamic editable projectable generator basis

In [ ]:
# ============================================================================
# Cell 4 — Dynamic editable projectable generator basis
# ============================================================================

BASIS_LIBRARY = []


def add_basis(name, source, fn):
    BASIS_LIBRARY.append({"name": name, "source": source, "fn": fn})


def time_basis(t):
    return t / CONFIG["INPUT_T_SCALE"]


def safe_exp(z):
    clip = CONFIG["BASIS_EXP_CLIP"]
    return jnp.exp(jnp.clip(z, -clip, clip))


def zero_xi():
    return jnp.zeros((D,), dtype=jnp.float32)


def e_i(i):
    return jnp.zeros((D,), dtype=jnp.float32).at[i].set(1.0)


def unit(v):
    return v / (jnp.linalg.norm(v) + 1e-8)


def make_low_rank_directions(k_requested):
    # At most D mutually distinct useful coordinate-index directions are used.
    k = min(int(k_requested), D)
    idx = jnp.arange(D, dtype=jnp.float32)
    directions = [unit(jnp.ones((D,), dtype=jnp.float32))]
    freq = 1
    while len(directions) < k:
        phase = 2.0 * jnp.pi * freq * (idx + 0.5) / float(max(D, 1))
        directions.append(unit(jnp.sin(phase)))
        if len(directions) < k:
            directions.append(unit(jnp.cos(phase)))
        freq += 1
    return tuple(directions[:k])


# Time-only generators.
add_basis(
    "time_translation__dt",
    "Generic projectable time translation",
    lambda t, x: jnp.concatenate([
        jnp.array([1.0], dtype=jnp.float32), zero_xi()
    ]),
)
add_basis(
    "time_scaling__t_dt",
    "Generic projectable time scaling",
    lambda t, x: jnp.concatenate([
        jnp.array([time_basis(t)], dtype=jnp.float32), zero_xi()
    ]),
)
add_basis(
    "time_quadratic__t2_dt",
    "Generic projectable quadratic time generator",
    lambda t, x: jnp.concatenate([
        jnp.array([time_basis(t) ** 2], dtype=jnp.float32), zero_xi()
    ]),
)

# Coordinate translations and coordinate scalings.
for i in range(D):
    add_basis(
        f"translation__dx{i+1}",
        "Generic coordinate translation",
        lambda t, x, i=i: jnp.concatenate([
            jnp.array([0.0], dtype=jnp.float32), e_i(i)
        ]),
    )
    add_basis(
        f"scaling__x{i+1}_dx{i+1}",
        "Generic coordinate scaling",
        lambda t, x, i=i: jnp.concatenate([
            jnp.array([0.0], dtype=jnp.float32),
            jnp.zeros((D,), dtype=jnp.float32).at[i].set(x[i]),
        ]),
    )

# Lifted Brownian/Gaeta scaling.
add_basis(
    "global_brownian_scaling__2t_dt_plus_x_dot_dx",
    "Lifted Brownian/Gaeta scaling",
    lambda t, x: jnp.concatenate([
        jnp.array([2.0 * time_basis(t)], dtype=jnp.float32), x
    ]),
)

# Exponential Gaeta-style terms.
if CONFIG["BASIS_ADD_EXPONENTIAL"]:
    for mu_value in CONFIG["BASIS_LINEAR_MU_LIST"]:
        mu = float(mu_value)
        if abs(mu) < 1e-12:
            continue

        for i in range(D):
            add_basis(
                f"exp_translation__mu_{mu:+.2f}__dx{i+1}",
                "Gaeta-style exponential translation",
                lambda t, x, mu=mu, i=i: jnp.concatenate([
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.zeros((D,), dtype=jnp.float32).at[i].set(
                        safe_exp(mu * time_basis(t))
                    ),
                ]),
            )

        add_basis(
            f"exp_projective__mu_{mu:+.2f}",
            "Gaeta-style exponential projective generator",
            lambda t, x, mu=mu: jnp.concatenate([
                jnp.array(
                    [safe_exp(2.0 * mu * time_basis(t))],
                    dtype=jnp.float32,
                ),
                mu * safe_exp(2.0 * mu * time_basis(t)) * x,
            ]),
        )

LOW_RANK_DIRECTIONS = make_low_rank_directions(CONFIG["BASIS_LOW_RANK_K"])
K_LOW = len(LOW_RANK_DIRECTIONS)

if CONFIG["BASIS_ADD_LOW_RANK_ROTATIONS"]:
    for a in range(K_LOW):
        for b in range(a + 1, K_LOW):
            va = LOW_RANK_DIRECTIONS[a]
            vb = LOW_RANK_DIRECTIONS[b]
            add_basis(
                f"lowrank_rotation__v{a+1}_v{b+1}",
                "Low-rank antisymmetric rotation",
                lambda t, x, va=va, vb=vb: jnp.concatenate([
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.dot(vb, x) * va - jnp.dot(va, x) * vb,
                ]),
            )

if CONFIG["BASIS_ADD_LOW_RANK_SHEARS"]:
    for a in range(K_LOW - 1):
        va = LOW_RANK_DIRECTIONS[a]
        vb = LOW_RANK_DIRECTIONS[a + 1]
        add_basis(
            f"lowrank_shear__v{a+2}_to_v{a+1}",
            "Low-rank adjacent shear",
            lambda t, x, va=va, vb=vb: jnp.concatenate([
                jnp.array([0.0], dtype=jnp.float32), jnp.dot(vb, x) * va
            ]),
        )
        add_basis(
            f"lowrank_shear__v{a+1}_to_v{a+2}",
            "Low-rank adjacent shear",
            lambda t, x, va=va, vb=vb: jnp.concatenate([
                jnp.array([0.0], dtype=jnp.float32), jnp.dot(va, x) * vb
            ]),
        )

BASIS_NAMES = [item["name"] for item in BASIS_LIBRARY]
BASIS_SOURCES = [item["source"] for item in BASIS_LIBRARY]
BASIS_FNS = tuple(item["fn"] for item in BASIS_LIBRARY)
N_BASIS = len(BASIS_FNS)

print("\nBasis library")
print("  dimension              :", D)
print("  low-rank directions    :", K_LOW)
print("  number of basis terms  :", N_BASIS)


def eval_basis_one(t, x):
    return jnp.stack([fn(t, x) for fn in BASIS_FNS], axis=0)


def eval_generators_point(W, z):
    return W @ eval_basis_one(z[0], z[1:])


def eval_generators_batch(W, tx):
    return jax.vmap(lambda z: eval_generators_point(W, z))(tx)


def sample_tx_data_jitter(key, dataset, n_points, jitter_std):
    key_idx, key_noise = jax.random.split(key)
    t0, x0, _, _ = dataset
    idx = jax.random.randint(key_idx, (n_points,), 0, t0.shape[0])
    x = x0[idx] + jitter_std * jax.random.normal(key_noise, (n_points, D))
    return jnp.concatenate([t0[idx, None], x], axis=1)


def sample_tx_box(key, n_points):
    key_t, key_x = jax.random.split(key)
    t = jax.random.uniform(
        key_t, (n_points, 1), minval=0.0, maxval=CONFIG["T_FINAL"]
    )
    x = jax.random.uniform(
        key_x,
        (n_points, D),
        minval=-CONFIG["INPUT_X_SCALE"],
        maxval=CONFIG["INPUT_X_SCALE"],
    )
    return jnp.concatenate([t, x], axis=1)


def sample_e8_collocation(key, n_points, dataset, mode):
    if mode == "data_jitter":
        return sample_tx_data_jitter(
            key, dataset, n_points, CONFIG["STAGE_F_JITTER_STD"]
        )
    if mode == "box_uniform":
        return sample_tx_box(key, n_points)
    if mode == "trajectory":
        t, x, _, _ = sample_batch(key, dataset, n_points)
        return jnp.concatenate([t[:, None], x], axis=1)
    raise ValueError(f"Unknown collocation mode: {mode}")


key, subkey = jax.random.split(key)
W = CONFIG["SYM_INIT_SCALE"] * jax.random.normal(
    subkey, (CONFIG["N_GENERATORS"], N_BASIS)
)
print("  initial W shape        :", W.shape)


## Cell 5 — Raw/normalized residuals, closure, and Stage-II discovery

In [ ]:
# ============================================================================
# Cell 5 — Raw/normalized residuals, closure, and Stage-II discovery
# ============================================================================

def det_residuals_point(sde_params, W, z):
    """Projectable Itô determining residuals for diagonal diffusion."""
    f = f_vec_single(sde_params, z)
    sigma = sigma_vec_single(sde_params, z)

    J_f = jax.jacfwd(lambda zz: f_vec_single(sde_params, zz))(z)
    J_sigma = jax.jacfwd(lambda zz: sigma_vec_single(sde_params, zz))(z)

    f_t, f_x = J_f[:, 0], J_f[:, 1:]
    sigma_t, sigma_x = J_sigma[:, 0], J_sigma[:, 1:]

    generator_fn = lambda zz: eval_generators_point(W, zz)
    X = generator_fn(z)
    J_X = jax.jacfwd(generator_fn)(z)

    tau = X[:, 0]
    xi = X[:, 1:]
    tau_t = J_X[:, 0, 0]
    tau_x = J_X[:, 0, 1:]
    xi_t = J_X[:, 1:, 0]
    xi_x = J_X[:, 1:, 1:]

    # All basis fields used here are at most linear in x, hence xi_xx = 0.
    adv_xi = jnp.einsum("j,aij->ai", f, xi_x)
    xi_dot_grad_f = jnp.einsum("aj,ij->ai", xi, f_x)
    R_f = (
        xi_t
        + adv_xi
        - xi_dot_grad_f
        - tau[:, None] * f_t[None, :]
        - tau_t[:, None] * f[None, :]
    )

    eye = jnp.eye(D, dtype=jnp.float32)
    first_term = jnp.einsum("p,aip->aip", sigma, xi_x)
    diagonal_correction = (
        jnp.einsum("aj,ij->ai", xi, sigma_x)
        + tau[:, None] * sigma_t[None, :]
        + 0.5 * tau_t[:, None] * sigma[None, :]
    )
    R_sigma = first_term - diagonal_correction[:, :, None] * eye[None, :, :]
    return R_f, R_sigma, tau_x


def det_residual_components(sde_params, W, tx):
    """
    Return both raw sums and dimension-normalized determining losses.

    Averages are over collocation points and learned generators. The raw
    quantities retain sums over physical residual entries.
    """
    R_f, R_sigma, tau_x = jax.vmap(
        lambda z: det_residuals_point(sde_params, W, z)
    )(tx)

    drift_raw = jnp.mean(jnp.sum(R_f**2, axis=-1))
    diffusion_raw = jnp.mean(jnp.sum(R_sigma**2, axis=(-2, -1)))

    drift_per_component = drift_raw / float(D)
    diffusion_per_full_entry = diffusion_raw / float(D * D)
    diffusion_per_diagonal_entry = diffusion_raw / float(D)

    normalized_total = drift_per_component + diffusion_per_full_entry
    projectability = jnp.mean(tau_x**2)

    return {
        "drift_raw": drift_raw,
        "diffusion_raw": diffusion_raw,
        "drift_per_component": drift_per_component,
        "diffusion_per_full_entry": diffusion_per_full_entry,
        "diffusion_per_diagonal_entry": diffusion_per_diagonal_entry,
        "normalized_total": normalized_total,
        "projectability": projectability,
    }


def determining_loss(sde_params, W, tx):
    components = det_residual_components(sde_params, W, tx)
    return components["normalized_total"], components["projectability"]


def generator_jacobians_batch(W, tx):
    def one(z):
        fn = lambda zz: eval_generators_point(W, zz)
        return fn(z), jax.jacfwd(fn)(z)
    return jax.vmap(one)(tx)


def lie_closure_loss_global(W, tx):
    m = W.shape[0]
    if m < 2:
        return jnp.array(0.0, dtype=jnp.float32)

    X, J = generator_jacobians_batch(W, tx)
    S = tx.shape[0]
    idx_i, idx_j = jnp.triu_indices(m, k=1)
    P = idx_i.shape[0]

    Xi, Xj = X[:, idx_i, :], X[:, idx_j, :]
    Ji, Jj = J[:, idx_i, :, :], J[:, idx_j, :, :]
    brackets = (
        jnp.einsum("spc,spqc->spq", Xi, Jj)
        - jnp.einsum("spc,spqc->spq", Xj, Ji)
    )

    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T
    B = jnp.transpose(brackets, (1, 0, 2)).reshape(P, -1).T
    G = V.T @ V
    reg = (CONFIG["CLOSURE_RCOND"] ** 2) * jnp.eye(m, dtype=G.dtype)
    C = jnp.linalg.solve(G + reg, V.T @ B)
    return jnp.mean((B - V @ C) ** 2)


def gram_independence_loss(W, tx):
    X = eval_generators_batch(W, tx)
    S, m = X.shape[0], X.shape[1]
    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T
    G = (V.T @ V) / float(Q * S)
    identity = jnp.eye(m, dtype=G.dtype)
    independence = jnp.mean((G - identity) ** 2)
    norm_hinge = jnp.mean(
        jnp.maximum(0.0, CONFIG["MIN_GENERATOR_NORM"] - jnp.diag(G)) ** 2
    )
    return independence, norm_hinge, G


def sym_discovery_objective(W, sde_params, tx):
    det, projectable = determining_loss(sde_params, W, tx)
    closure = lie_closure_loss_global(W, tx)
    independence, norm_hinge, _ = gram_independence_loss(W, tx)
    total = (
        CONFIG["LAMBDA_DET"] * det
        + CONFIG["LAMBDA_CLOSE"] * closure
        + CONFIG["LAMBDA_IND"] * independence
        + CONFIG["LAMBDA_PROJECTABLE"] * projectable
        + CONFIG["LAMBDA_NORM_HINGE"] * norm_hinge
    )
    return total, {
        "total": total,
        "det": det,
        "closure": closure,
        "independence": independence,
        "projectable": projectable,
        "norm_hinge": norm_hinge,
    }


@jax.jit
def sym_train_step(W, opt_state, sde_params, tx):
    loss_fn = lambda W_local: sym_discovery_objective(W_local, sde_params, tx)[0]
    _, grads = jax.value_and_grad(loss_fn)(W)
    W, opt_state = adam_update(W, grads, opt_state, CONFIG["SYM_DISCOVERY_LR"])
    _, aux = sym_discovery_objective(W, sde_params, tx)
    return W, opt_state, aux


W_opt_state = adam_init(W)
sym_history = []
start = time.time()

for epoch in range(1, CONFIG["SYM_DISCOVERY_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    tx_disc = sample_tx_data_jitter(
        subkey,
        train_data,
        CONFIG["SYM_DISCOVERY_POINTS"],
        CONFIG["SYM_JITTER_STD"],
    )
    W, W_opt_state, aux = sym_train_step(
        W, W_opt_state, sde_params_baseline, tx_disc
    )

    if epoch == 1 or epoch % CONFIG["PRINT_EVERY"] == 0:
        aux_np = {name: float(value) for name, value in aux.items()}
        sym_history.append(aux_np)
        print(
            f"[Stage II] epoch {epoch:5d} | "
            f"total {aux_np['total']: .4e} | "
            f"det {aux_np['det']: .4e} | "
            f"closure {aux_np['closure']: .4e} | "
            f"ind {aux_np['independence']: .4e}"
        )

symmetry_discovery_runtime_sec = time.time() - start
print(f"Stage-II runtime: {symmetry_discovery_runtime_sec:.2f} sec")

W_np = np.asarray(jax.device_get(W))
for generator_id in range(CONFIG["N_GENERATORS"]):
    order = np.argsort(-np.abs(W_np[generator_id]))[:10]
    print(f"\nGenerator X_{generator_id + 1}: top coefficients")
    for basis_id in order:
        print(
            f"  {basis_id:4d} | {W_np[generator_id, basis_id]: .6f} | "
            f"{BASIS_NAMES[basis_id]}"
        )



## Cell 6 — Paired Stage-III branches and gradient diagnostics

In [ ]:
# ============================================================================
# Cell 6 — Paired Stage-III branches and gradient diagnostics
# ============================================================================

def stage_f_objective(params, W_fixed, batch, tx_extra, lambda_sym):
    data_loss = data_nll_loss(params, batch)
    t_batch, x_batch, _, _ = batch
    n_data_sym = min(CONFIG["STAGE_F_N_DATA_SYM_POINTS"], t_batch.shape[0])
    tx_data = jnp.concatenate(
        [t_batch[:n_data_sym, None], x_batch[:n_data_sym]], axis=1
    )
    tx_sym = jnp.concatenate([tx_data, tx_extra], axis=0)
    components = det_residual_components(params, W_fixed, tx_sym)
    total = (
        data_loss
        + lambda_sym * components["normalized_total"]
        + CONFIG["STAGE_F_LAMBDA_PROJECTABLE"] * components["projectability"]
    )
    return total, {
        "total": total,
        "data": data_loss,
        **components,
    }


def make_stage_f_train_step(lambda_sym):
    lambda_sym = float(lambda_sym)

    @jax.jit
    def train_step(params, opt_state, W_fixed, batch, tx_extra):
        objective = lambda p: stage_f_objective(
            p, W_fixed, batch, tx_extra, lambda_sym
        )[0]
        _, grads = jax.value_and_grad(objective)(params)
        grad_norm = tree_l2_norm(grads)
        params, opt_state = adam_update(
            params, grads, opt_state, CONFIG["STAGE_F_LR"]
        )
        _, aux = stage_f_objective(
            params, W_fixed, batch, tx_extra, lambda_sym
        )
        return params, opt_state, aux, grad_norm

    return train_step


def e8_gradient_diagnostics(params, W_fixed, data_batch, tx_sym, lambda_sym):
    grad_data = jax.grad(lambda p: data_nll_loss(p, data_batch))(params)
    grad_drift_raw = jax.grad(
        lambda p: det_residual_components(p, W_fixed, tx_sym)["drift_raw"]
    )(params)
    grad_diffusion_raw = jax.grad(
        lambda p: det_residual_components(p, W_fixed, tx_sym)["diffusion_raw"]
    )(params)
    grad_normalized = jax.grad(
        lambda p: det_residual_components(p, W_fixed, tx_sym)["normalized_total"]
    )(params)

    data_norm = tree_l2_norm(grad_data)
    drift_raw_norm = tree_l2_norm(grad_drift_raw)
    diffusion_raw_norm = tree_l2_norm(grad_diffusion_raw)
    normalized_norm = tree_l2_norm(grad_normalized)

    return {
        "grad_data_norm": data_norm,
        "grad_drift_raw_norm": drift_raw_norm,
        "grad_diffusion_raw_norm": diffusion_raw_norm,
        "grad_normalized_symmetry_norm": normalized_norm,
        "unweighted_sym_to_data_ratio": normalized_norm / (data_norm + 1e-12),
        "weighted_sym_to_data_ratio": (
            float(lambda_sym) * normalized_norm / (data_norm + 1e-12)
        ),
    }


def train_stage_f_branch(branch_name, lambda_sym, shared_batch_keys, shared_extra_keys):
    params = sde_params_baseline
    opt_state = adam_init(params)
    train_step = make_stage_f_train_step(lambda_sym)
    history = []
    max_gradient_norm = 0.0
    nonfinite_steps = 0
    step_times = []

    # Warm-up compiles this branch before timing the real loop.
    warm_batch = sample_batch(
        shared_batch_keys[0], train_data, CONFIG["STAGE_F_BATCH_SIZE"]
    )
    warm_extra = sample_e8_collocation(
        shared_extra_keys[0],
        CONFIG["STAGE_F_N_EXTRA_SYM_POINTS"],
        train_data,
        CONFIG["E8_EVAL_COLLOCATION_MODE"],
    )
    _ = train_step(params, opt_state, W, warm_batch, warm_extra)
    jax.block_until_ready(_[2]["total"])

    start = time.time()
    for epoch in range(1, CONFIG["STAGE_F_EPOCHS"] + 1):
        batch = sample_batch(
            shared_batch_keys[epoch - 1],
            train_data,
            CONFIG["STAGE_F_BATCH_SIZE"],
        )
        tx_extra = sample_e8_collocation(
            shared_extra_keys[epoch - 1],
            CONFIG["STAGE_F_N_EXTRA_SYM_POINTS"],
            train_data,
            CONFIG["E8_EVAL_COLLOCATION_MODE"],
        )

        step_start = time.time()
        params, opt_state, aux, grad_norm = train_step(
            params, opt_state, W, batch, tx_extra
        )
        jax.block_until_ready(aux["total"])
        step_times.append(time.time() - step_start)

        grad_norm_float = float(grad_norm)
        max_gradient_norm = max(max_gradient_norm, grad_norm_float)
        finite = bool(np.isfinite(float(aux["total"]))) and bool(
            np.isfinite(grad_norm_float)
        )
        if not finite:
            nonfinite_steps += 1
            print(f"[{branch_name}] stopped at epoch {epoch}: non-finite value.")
            break

        if epoch == 1 or epoch % CONFIG["PRINT_EVERY"] == 0:
            aux_np = {name: float(value) for name, value in aux.items()}
            history.append(aux_np)
            print(
                f"[{branch_name}] epoch {epoch:5d} | "
                f"total {aux_np['total']: .6e} | "
                f"data {aux_np['data']: .6e} | "
                f"norm-sym {aux_np['normalized_total']: .6e} | "
                f"valid {float(data_nll_loss(params, valid_data)): .6f}"
            )

    runtime_sec = time.time() - start
    return {
        "params": params,
        "lambda_sym": float(lambda_sym),
        "loss_mode": "per_entry_determining_loss",
        "runtime_sec": float(runtime_sec),
        "mean_step_sec": float(np.mean(step_times)) if step_times else np.nan,
        "nonfinite_steps": int(nonfinite_steps),
        "max_gradient_norm": float(max_gradient_norm),
        "history": history,
    }


# All branches receive exactly the same batch and collocation sequences.
key, key_batches, key_extras = jax.random.split(key, 3)
shared_batch_keys = jax.random.split(key_batches, CONFIG["STAGE_F_EPOCHS"])
shared_extra_keys = jax.random.split(key_extras, CONFIG["STAGE_F_EPOCHS"])

stage_f_runs = {}

# Submitted reference branch. It uses the same per-entry determining
# objective as the sweep branches.
if CONFIG["E8_RUN_LEGACY_BRANCH"]:
    submitted_branch_name = (
        f"per_entry_lambda_{CONFIG['E8_LEGACY_LAMBDA']:g}"
    )

    submitted_run = train_stage_f_branch(
        submitted_branch_name,
        CONFIG["E8_LEGACY_LAMBDA"],
        shared_batch_keys,
        shared_extra_keys,
    )

    submitted_run["loss_mode"] = "per_entry_determining_loss"
    submitted_run["is_submitted_reference"] = True

    stage_f_runs[submitted_branch_name] = submitted_run


# Remaining per-entry lambda sweep.
for lambda_value in CONFIG["E8_NORMALIZED_LAMBDAS"]:
    lambda_value = float(lambda_value)

    # Avoid accidentally training an exact duplicate of the submitted branch.
    if (
        CONFIG["E8_RUN_LEGACY_BRANCH"]
        and abs(lambda_value - float(CONFIG["E8_LEGACY_LAMBDA"])) < 1e-12
    ):
        continue

    branch_name = f"per_entry_lambda_{lambda_value:g}"

    branch_run = train_stage_f_branch(
        branch_name,
        lambda_value,
        shared_batch_keys,
        shared_extra_keys,
    )

    branch_run["loss_mode"] = "per_entry_determining_loss"
    branch_run["is_submitted_reference"] = False

    stage_f_runs[branch_name] = branch_run


print("\nCompleted Stage-III branches:")

for branch_name, branch_info in stage_f_runs.items():
    if branch_info.get("is_submitted_reference", False):
        marker = " [submitted reference]"
    else:
        marker = ""

    print(
        f"  {branch_name:<32s} "
        f"lambda={branch_info['lambda_sym']:<10g} "
        f"runtime={branch_info['runtime_sec']:.2f}s "
        f"nonfinite={branch_info['nonfinite_steps']}"
        f"{marker}"
    )


## Cell 7 — Evaluation helper functions used by the final report

In [ ]:
# ============================================================================
# Cell 7 — Evaluation helper functions used by the final report
# ============================================================================

def _to_float(x):
    return float(np.asarray(jax.device_get(x)))


def _safe_ratio(num, den, eps=1e-12):
    return float(num) / (abs(float(den)) + eps)


def _safe_pct_reduction(new, old, eps=1e-12):
    return 100.0 * (float(old) - float(new)) / (abs(float(old)) + eps)


def _summary_stats(values):
    values = np.asarray(values, dtype=np.float64)
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
    return {
        "mean": mean,
        "std": std,
        "variance": float(std**2),
        "sem": float(std / np.sqrt(max(len(values), 1))),
        "cv": float(std / (abs(mean) + 1e-12)),
        "minimum": float(np.min(values)),
        "maximum": float(np.max(values)),
    }


def _coefficient_metrics(params, tx):
    t, x = tx[:, 0], tx[:, 1:]
    f_true = jax.vmap(true_drift)(t, x)
    sigma_true = jax.vmap(true_sigma)(t, x)
    f_pred, sigma_pred = batched_surrogate_values(params, t, x)

    drift_error = f_pred - f_true
    sigma_error = sigma_pred - sigma_true
    drift_mse = jnp.mean(drift_error**2)
    sigma_mse = jnp.mean(sigma_error**2)

    return {
        "drift_mse": _to_float(drift_mse),
        "drift_rmse": _to_float(jnp.sqrt(drift_mse)),
        "drift_rel_l2": _to_float(
            jnp.linalg.norm(drift_error) / (jnp.linalg.norm(f_true) + 1e-12)
        ),
        "sigma_mse": _to_float(sigma_mse),
        "sigma_rmse": _to_float(jnp.sqrt(sigma_mse)),
        "sigma_rel_l2": _to_float(
            jnp.linalg.norm(sigma_error) / (jnp.linalg.norm(sigma_true) + 1e-12)
        ),
    }


def _residual_metrics(params, tx):
    return {
        name: _to_float(value)
        for name, value in det_residual_components(params, W, tx).items()
    }


def _one_step_metrics(params, dataset):
    t, x, dt, dx = dataset
    f_pred, sigma_pred = batched_surrogate_values(params, t, x)
    predicted_mean = f_pred * dt[:, None]
    predicted_var = sigma_pred**2 * dt[:, None] + 1e-8
    error = dx - predicted_mean
    return {
        "increment_rmse": _to_float(jnp.sqrt(jnp.mean(error**2))),
        "increment_mae": _to_float(jnp.mean(jnp.abs(error))),
        "standardized_sq_residual": _to_float(
            jnp.mean(error**2 / predicted_var)
        ),
    }


def _resolve_dataset_split(split_name):
    split_name = str(split_name).lower()
    if split_name == "train":
        return train_data
    if split_name in {"valid", "validation"}:
        return valid_data
    if split_name == "test":
        return test_data
    raise ValueError(
        f"Unknown diagnostic split {split_name!r}; expected train, valid, or test."
    )


## Cell 8 — Complete E8 numerical report and aggregation

In [ ]:
# ============================================================================
# Cell 8 — Complete E8 numerical report and aggregation
# ============================================================================

branches = {
    "baseline": {
        "params": sde_params_baseline,
        "lambda_sym": 0.0,
        "loss_mode": "data_only",
        "runtime_sec": 0.0,
        "mean_step_sec": 0.0,
        "nonfinite_steps": 0,
        "max_gradient_norm": np.nan,
        "is_submitted_reference": False,
    },
    **stage_f_runs,
}

# Use one common split for the fixed residual and Monte Carlo diagnostics.
residual_diagnostic_data = _resolve_dataset_split(
    CONFIG["E8_RESIDUAL_DIAGNOSTIC_SPLIT"]
)

# -----------------------------------------------------------------------------
# Fixed evaluation clouds
# -----------------------------------------------------------------------------

eval_key = jax.random.PRNGKey(CONFIG["SEED"] + 910000 + D)
eval_key, key_coeff, key_residual = jax.random.split(eval_key, 3)

tx_coeff_eval = sample_e8_collocation(
    key_coeff,
    CONFIG["N_EVAL_POINTS"],
    test_data,
    CONFIG["E8_EVAL_COLLOCATION_MODE"],
)

tx_residual_eval = sample_e8_collocation(
    key_residual,
    CONFIG["E8_RESIDUAL_EVAL_POINTS"],
    residual_diagnostic_data,
    CONFIG["E8_EVAL_COLLOCATION_MODE"],
)

# -----------------------------------------------------------------------------
# Exact same-batch implementation consistency check
# -----------------------------------------------------------------------------

residual_consistency_check = None

if CONFIG["E8_RUN_RESIDUAL_CONSISTENCY_CHECK"]:
    consistency_key = jax.random.PRNGKey(CONFIG["SEED"] + 915000 + D)

    tx_consistency = sample_e8_collocation(
        consistency_key,
        CONFIG["E8_RESIDUAL_CHECK_POINTS"],
        residual_diagnostic_data,
        CONFIG["E8_EVAL_COLLOCATION_MODE"],
    )

    wrapped_metrics = _residual_metrics(
        sde_params_baseline,
        tx_consistency,
    )

    direct_metrics = {
        name: _to_float(value)
        for name, value in det_residual_components(
            sde_params_baseline,
            W,
            tx_consistency,
        ).items()
    }

    residual_consistency_check = {}

    for metric_name, wrapped_value in wrapped_metrics.items():
        direct_value = direct_metrics[metric_name]
        absolute_difference = abs(
            wrapped_value - direct_value
        )

        np.testing.assert_allclose(
            wrapped_value,
            direct_value,
            rtol=CONFIG["E8_RESIDUAL_CHECK_RTOL"],
            atol=CONFIG["E8_RESIDUAL_CHECK_ATOL"],
            err_msg=(
                f"Residual consistency failure for {metric_name}"
            ),
        )

        residual_consistency_check[metric_name] = {
            "wrapped": wrapped_value,
            "direct": direct_value,
            "absolute_difference": absolute_difference,
        }

    tx_consistency_np = np.asarray(
        jax.device_get(tx_consistency)
    )

    residual_consistency_check[
        "collocation_statistics"
    ] = {
        "t_min": float(
            tx_consistency_np[:, 0].min()
        ),
        "t_max": float(
            tx_consistency_np[:, 0].max()
        ),
        "x_mean": float(
            tx_consistency_np[:, 1:].mean()
        ),
        "x_std": float(
            tx_consistency_np[:, 1:].std()
        ),
        "x_min": float(
            tx_consistency_np[:, 1:].min()
        ),
        "x_max": float(
            tx_consistency_np[:, 1:].max()
        ),
    }

# -----------------------------------------------------------------------------
# Final branch metrics
# -----------------------------------------------------------------------------

branch_results = {}

for branch_name, branch_info in branches.items():
    params = branch_info["params"]

    coefficient_metrics = _coefficient_metrics(
        params,
        tx_coeff_eval,
    )

    residual_metrics = _residual_metrics(
        params,
        tx_residual_eval,
    )

    train_step_metrics = _one_step_metrics(
        params,
        train_data,
    )

    valid_step_metrics = _one_step_metrics(
        params,
        valid_data,
    )

    test_step_metrics = _one_step_metrics(
        params,
        test_data,
    )

    branch_results[branch_name] = {
        "train_nll": _to_float(
            data_nll_loss(params, train_data)
        ),
        "valid_nll": _to_float(
            data_nll_loss(params, valid_data)
        ),
        "test_nll": _to_float(
            data_nll_loss(params, test_data)
        ),

        **coefficient_metrics,

        **{
            f"residual_{name}": value
            for name, value
            in residual_metrics.items()
        },

        **{
            f"train_{name}": value
            for name, value
            in train_step_metrics.items()
        },

        **{
            f"valid_{name}": value
            for name, value
            in valid_step_metrics.items()
        },

        **{
            f"test_{name}": value
            for name, value
            in test_step_metrics.items()
        },

        "lambda_sym": float(
            branch_info.get(
                "lambda_sym",
                np.nan,
            )
        ),
        "loss_mode": str(
            branch_info.get(
                "loss_mode",
                "unknown",
            )
        ),
        "runtime_sec": float(
            branch_info.get(
                "runtime_sec",
                np.nan,
            )
        ),
        "mean_step_sec": float(
            branch_info.get(
                "mean_step_sec",
                np.nan,
            )
        ),
        "nonfinite_steps": int(
            branch_info.get(
                "nonfinite_steps",
                0,
            )
        ),
        "max_gradient_norm": float(
            branch_info.get(
                "max_gradient_norm",
                np.nan,
            )
        ),
        "is_submitted_reference": bool(
            branch_info.get(
                "is_submitted_reference",
                False,
            )
        ),
    }

# -----------------------------------------------------------------------------
# Common-initialization and final-parameter gradient diagnostics
# -----------------------------------------------------------------------------

grad_key = jax.random.PRNGKey(
    CONFIG["SEED"] + 920000 + D
)

grad_key, key_grad_batch, key_grad_tx = (
    jax.random.split(
        grad_key,
        3,
    )
)

grad_batch = sample_batch(
    key_grad_batch,
    train_data,
    CONFIG["E8_GRAD_DATA_BATCH_SIZE"],
)

grad_tx = sample_e8_collocation(
    key_grad_tx,
    CONFIG["E8_GRAD_COLLOCATION_POINTS"],
    train_data,
    CONFIG["E8_EVAL_COLLOCATION_MODE"],
)

# The unweighted initialization gradients are identical for every branch.
# Compute them once and change only the weighted ratio for each lambda.
initial_base = {
    name: _to_float(value)
    for name, value in e8_gradient_diagnostics(
        sde_params_baseline,
        W,
        grad_batch,
        grad_tx,
        0.0,
    ).items()
}

gradient_results_initial = {}
gradient_results_final = {}

for branch_name, branch_info in branches.items():
    lambda_sym = float(
        branch_info.get(
            "lambda_sym",
            0.0,
        )
    )

    initial_row = dict(initial_base)

    initial_row[
        "weighted_sym_to_data_ratio"
    ] = (
        lambda_sym
        * initial_row[
            "unweighted_sym_to_data_ratio"
        ]
    )

    gradient_results_initial[
        branch_name
    ] = initial_row

    final_diagnostics = e8_gradient_diagnostics(
        branch_info["params"],
        W,
        grad_batch,
        grad_tx,
        lambda_sym,
    )

    gradient_results_final[
        branch_name
    ] = {
        name: _to_float(value)
        for name, value
        in final_diagnostics.items()
    }

# -----------------------------------------------------------------------------
# Monte Carlo residual-estimator variance
# -----------------------------------------------------------------------------

mc_batch_sizes = list(
    CONFIG["E8_MC_BATCH_SIZES"]
)

mc_repeats = int(
    CONFIG["E8_MC_REPEATS"]
)

mc_reference_branch = str(
    CONFIG["E8_MC_REFERENCE_BRANCH"]
)

if mc_reference_branch not in branches:
    raise ValueError(
        f"E8_MC_REFERENCE_BRANCH="
        f"{mc_reference_branch!r} is unavailable. "
        f"Available branches: "
        f"{list(branches.keys())}"
    )

mc_params = branches[
    mc_reference_branch
]["params"]

mc_rows = []

mc_master_key = jax.random.PRNGKey(
    CONFIG["SEED"] + 930000 + D
)

for batch_size in mc_batch_sizes:
    samples = {
        "drift_raw": [],
        "diffusion_raw": [],
        "drift_per_component": [],
        "diffusion_per_full_entry": [],
        "diffusion_per_diagonal_entry": [],
        "normalized_total": [],
    }

    repeat_keys = jax.random.split(
        jax.random.fold_in(
            mc_master_key,
            int(batch_size),
        ),
        mc_repeats,
    )

    for repeat_key in repeat_keys:
        tx_mc = sample_e8_collocation(
            repeat_key,
            int(batch_size),
            residual_diagnostic_data,
            CONFIG[
                "E8_VARIANCE_COLLOCATION_MODE"
            ],
        )

        metrics = _residual_metrics(
            mc_params,
            tx_mc,
        )

        for metric_name in samples:
            samples[
                metric_name
            ].append(
                metrics[metric_name]
            )

    for metric_name, values in samples.items():
        mc_rows.append({
            "dimension": int(D),
            "reference_branch": (
                mc_reference_branch
            ),
            "diagnostic_split": (
                CONFIG[
                    "E8_RESIDUAL_DIAGNOSTIC_SPLIT"
                ]
            ),
            "collocation_mode": (
                CONFIG[
                    "E8_VARIANCE_COLLOCATION_MODE"
                ]
            ),
            "batch_size": int(
                batch_size
            ),
            "repeats": int(
                mc_repeats
            ),
            "metric": metric_name,
            **_summary_stats(values),
        })

# -----------------------------------------------------------------------------
# Deployable Stage-III selection with baseline abstention
# -----------------------------------------------------------------------------

refined_candidates = [
    branch_name
    for branch_name, branch_info
    in branches.items()
    if (
        branch_info.get(
            "loss_mode"
        )
        == "per_entry_determining_loss"
        and branch_results[
            branch_name
        ]["nonfinite_steps"] == 0
        and np.isfinite(
            branch_results[
                branch_name
            ]["valid_nll"]
        )
    )
]

if refined_candidates:
    best_refined_candidate = min(
        refined_candidates,
        key=lambda name: (
            branch_results[
                name
            ]["valid_nll"]
        ),
    )
else:
    best_refined_candidate = None

baseline_valid_nll = (
    branch_results[
        "baseline"
    ]["valid_nll"]
)

required_validation_gain = float(
    CONFIG[
        "E8_SELECTION_MIN_VALID_NLL_GAIN"
    ]
)

if best_refined_candidate is None:
    selected_branch = "baseline"

    selection_reason = (
        "No finite Stage-III candidate "
        "was available."
    )

else:
    candidate_valid_nll = (
        branch_results[
            best_refined_candidate
        ]["valid_nll"]
    )

    validation_nll_gain = (
        baseline_valid_nll
        - candidate_valid_nll
    )

    candidate_beats_baseline = (
        validation_nll_gain
        > required_validation_gain
    )

    if (
        not CONFIG[
            "E8_SELECTION_REQUIRE_BASELINE_IMPROVEMENT"
        ]
        or candidate_beats_baseline
    ):
        selected_branch = (
            best_refined_candidate
        )

        selection_reason = (
            "Selected the finite Stage-III "
            "candidate with the lowest validation "
            "NLL; it improved on the Stage-I "
            "baseline."
        )

    else:
        selected_branch = "baseline"

        selection_reason = (
            "Abstained from Stage III because "
            "no candidate improved validation "
            "NLL over the Stage-I baseline."
        )

selected_result = branch_results[
    selected_branch
]

baseline = branch_results[
    "baseline"
]

drift_raw_to_component_ratio = (
    _safe_ratio(
        baseline[
            "residual_drift_raw"
        ],
        baseline[
            "residual_drift_per_component"
        ],
    )
)

diffusion_raw_to_entry_ratio = (
    _safe_ratio(
        baseline[
            "residual_diffusion_raw"
        ],
        baseline[
            "residual_diffusion_per_full_entry"
        ],
    )
)

# -----------------------------------------------------------------------------
# Human-readable report
# -----------------------------------------------------------------------------

print(
    "\n"
    + "=" * 108
)

print(
    "E8 — DIMENSION AND "
    "DETERMINING-RESIDUAL SCALING"
)

print(
    "=" * 108
)

print(
    f"State dimension                         : "
    f"{D}"
)

print(
    f"Time-state coordinate dimension         : "
    f"{Q}"
)

print(
    f"SDE family                              : "
    f"{CONFIG['TRUE_SDE_NAME']}"
)

print(
    f"Seed                                    : "
    f"{CONFIG['SEED']}"
)

print(
    f"Learned generators                      : "
    f"{CONFIG['N_GENERATORS']}"
)

print(
    f"Basis templates                         : "
    f"{N_BASIS}"
)

print(
    f"Primary collocation mode                : "
    f"{CONFIG['E8_EVAL_COLLOCATION_MODE']}"
)

print(
    f"MC collocation mode                     : "
    f"{CONFIG['E8_VARIANCE_COLLOCATION_MODE']}"
)

print(
    f"Residual diagnostic split               : "
    f"{CONFIG['E8_RESIDUAL_DIAGNOSTIC_SPLIT']}"
)

# -----------------------------------------------------------------------------
# Section 1
# -----------------------------------------------------------------------------

print(
    "\n[1] BASELINE RAW AND "
    "DIMENSION-NORMALIZED RESIDUALS"
)

print(
    f"Raw drift residual                      : "
    f"{baseline['residual_drift_raw']:.8e}"
)

print(
    f"Drift residual per component            : "
    f"{baseline['residual_drift_per_component']:.8e}"
)

print(
    f"Raw/per-component ratio                 : "
    f"{drift_raw_to_component_ratio:.6f}"
)

print(
    f"Expected ratio d                        : "
    f"{D}"
)

print(
    f"Raw diffusion residual                  : "
    f"{baseline['residual_diffusion_raw']:.8e}"
)

print(
    f"Diffusion residual per full entry       : "
    f"{baseline['residual_diffusion_per_full_entry']:.8e}"
)

print(
    f"Raw/full-entry ratio                    : "
    f"{diffusion_raw_to_entry_ratio:.6f}"
)

print(
    f"Expected ratio d^2                      : "
    f"{D ** 2}"
)

print(
    f"Diffusion residual per diagonal entry   : "
    f"{baseline['residual_diffusion_per_diagonal_entry']:.8e}"
)

print(
    f"Normalized total residual               : "
    f"{baseline['residual_normalized_total']:.8e}"
)

print(
    f"Projectability residual                 : "
    f"{baseline['residual_projectability']:.8e} "
    f"(guaranteed by projectable basis)"
)

if residual_consistency_check is not None:
    print(
        "\nResidual implementation "
        "consistency check: PASS"
    )

    for metric_name, values in (
        residual_consistency_check.items()
    ):
        if (
            metric_name
            == "collocation_statistics"
        ):
            continue

        print(
            f"  {metric_name:<36s} "
            f"wrapped="
            f"{values['wrapped']:.8e} "
            f"direct="
            f"{values['direct']:.8e} "
            f"|difference|="
            f"{values['absolute_difference']:.3e}"
        )

    stats = residual_consistency_check[
        "collocation_statistics"
    ]

    print(
        "  collocation statistics                : "
        f"t=[{stats['t_min']:.4f}, "
        f"{stats['t_max']:.4f}], "
        f"x mean={stats['x_mean']:.4e}, "
        f"x std={stats['x_std']:.4e}, "
        f"x range=[{stats['x_min']:.4f}, "
        f"{stats['x_max']:.4f}]"
    )

# -----------------------------------------------------------------------------
# Section 2A
# -----------------------------------------------------------------------------

print(
    "\n[2A] COMMON-INITIALIZATION "
    "GRADIENT-SCALE DIAGNOSTICS"
)

print(
    "All rows use the same Stage-I "
    "parameters, data batch, and "
    "collocation batch."
)

for branch_name in branches:
    diagnostics = (
        gradient_results_initial[
            branch_name
        ]
    )

    branch_lambda = (
        branches[
            branch_name
        ]["lambda_sym"]
    )

    print(
        f"\n{branch_name}"
    )

    print(
        f"  lambda                                : "
        f"{branch_lambda:.8e}"
    )

    print(
        f"  ||grad data||                         : "
        f"{diagnostics['grad_data_norm']:.8e}"
    )

    print(
        f"  ||grad raw drift residual||           : "
        f"{diagnostics['grad_drift_raw_norm']:.8e}"
    )

    print(
        f"  ||grad raw diffusion residual||       : "
        f"{diagnostics['grad_diffusion_raw_norm']:.8e}"
    )

    print(
        f"  ||grad normalized symmetry||          : "
        f"{diagnostics['grad_normalized_symmetry_norm']:.8e}"
    )

    print(
        f"  unweighted sym/data ratio             : "
        f"{diagnostics['unweighted_sym_to_data_ratio']:.8e}"
    )

    print(
        f"  weighted sym/data ratio               : "
        f"{diagnostics['weighted_sym_to_data_ratio']:.8e}"
    )

# -----------------------------------------------------------------------------
# Section 2B
# -----------------------------------------------------------------------------

print(
    "\n[2B] FINAL-PARAMETER "
    "GRADIENT-SCALE DIAGNOSTICS"
)

print(
    "Each row uses that branch's final "
    "parameters and the same diagnostic "
    "batches."
)

for branch_name in branches:
    diagnostics = (
        gradient_results_final[
            branch_name
        ]
    )

    branch_lambda = (
        branches[
            branch_name
        ]["lambda_sym"]
    )

    print(
        f"\n{branch_name}"
    )

    print(
        f"  lambda                                : "
        f"{branch_lambda:.8e}"
    )

    print(
        f"  ||grad data||                         : "
        f"{diagnostics['grad_data_norm']:.8e}"
    )

    print(
        f"  ||grad raw drift residual||           : "
        f"{diagnostics['grad_drift_raw_norm']:.8e}"
    )

    print(
        f"  ||grad raw diffusion residual||       : "
        f"{diagnostics['grad_diffusion_raw_norm']:.8e}"
    )

    print(
        f"  ||grad normalized symmetry||          : "
        f"{diagnostics['grad_normalized_symmetry_norm']:.8e}"
    )

    print(
        f"  unweighted sym/data ratio             : "
        f"{diagnostics['unweighted_sym_to_data_ratio']:.8e}"
    )

    print(
        f"  weighted sym/data ratio               : "
        f"{diagnostics['weighted_sym_to_data_ratio']:.8e}"
    )

# -----------------------------------------------------------------------------
# Section 3
# -----------------------------------------------------------------------------

print(
    "\n[3] MONTE CARLO ESTIMATOR VARIANCE"
)

metrics_to_print = {
    "drift_raw",
    "diffusion_raw",
    "drift_per_component",
    "diffusion_per_full_entry",
    "normalized_total",
}

for batch_size in mc_batch_sizes:
    print(
        f"\nCollocation batch size "
        f"S={batch_size}"
    )

    for row in mc_rows:
        if (
            row["batch_size"] == batch_size
            and row["metric"]
            in metrics_to_print
        ):
            print(
                f"  {row['metric']:<34s} "
                f"mean={row['mean']:.8e} "
                f"std={row['std']:.8e} "
                f"var={row['variance']:.8e} "
                f"CV={row['cv']:.6f} "
                f"SEM={row['sem']:.8e}"
            )

# -----------------------------------------------------------------------------
# Section 4
# -----------------------------------------------------------------------------

print(
    "\n[4] FINAL BRANCH QUALITY "
    "AND STABILITY"
)

for branch_name, result in (
    branch_results.items()
):
    print(
        f"\n{branch_name}"
    )

    print(
        f"  loss mode                             : "
        f"{result['loss_mode']}"
    )

    print(
        f"  lambda                                : "
        f"{result['lambda_sym']:.8e}"
    )

    print(
        f"  submitted reference                   : "
        f"{result['is_submitted_reference']}"
    )

    print(
        f"  non-finite steps                      : "
        f"{result['nonfinite_steps']}"
    )

    if np.isfinite(
        result["max_gradient_norm"]
    ):
        print(
            f"  max pre-update gradient norm          : "
            f"{result['max_gradient_norm']:.8e}"
        )
    else:
        print(
            "  max pre-update gradient norm          : "
            "N/A (no Stage-III updates)"
        )

    print(
        f"  runtime sec                           : "
        f"{result['runtime_sec']:.6f}"
    )

    print(
        f"  mean step sec                         : "
        f"{result['mean_step_sec']:.8f}"
    )

    print(
        f"  train NLL                             : "
        f"{result['train_nll']:.8e}"
    )

    print(
        f"  validation NLL                        : "
        f"{result['valid_nll']:.8e}"
    )

    print(
        f"  test NLL                              : "
        f"{result['test_nll']:.8e}"
    )

    print(
        f"  drift MSE                             : "
        f"{result['drift_mse']:.8e}"
    )

    print(
        f"  diffusion MSE                         : "
        f"{result['sigma_mse']:.8e}"
    )

    print(
        f"  normalized residual                   : "
        f"{result['residual_normalized_total']:.8e}"
    )

    if branch_name != "baseline":
        drift_improvement = (
            _safe_pct_reduction(
                result["drift_mse"],
                baseline["drift_mse"],
            )
        )

        diffusion_improvement = (
            _safe_pct_reduction(
                result["sigma_mse"],
                baseline["sigma_mse"],
            )
        )

        residual_improvement = (
            _safe_pct_reduction(
                result[
                    "residual_normalized_total"
                ],
                baseline[
                    "residual_normalized_total"
                ],
            )
        )

        print(
            f"  drift MSE reduction vs baseline       : "
            f"{drift_improvement:.3f}%"
        )

        print(
            f"  diffusion MSE reduction vs baseline   : "
            f"{diffusion_improvement:.3f}%"
        )

        print(
            f"  normalized residual reduction         : "
            f"{residual_improvement:.3f}%"
        )

# -----------------------------------------------------------------------------
# Section 5
# -----------------------------------------------------------------------------

print(
    "\n[5] DEPLOYABLE STAGE-III "
    "SELECTION WITH ABSTENTION"
)

print(
    f"Selected branch                         : "
    f"{selected_branch}"
)

print(
    f"Selection reason                        : "
    f"{selection_reason}"
)

print(
    f"Selected lambda                         : "
    f"{selected_result['lambda_sym']:.8e}"
)

print(
    f"Selection validation NLL                : "
    f"{selected_result['valid_nll']:.8e}"
)

print(
    f"Held-out test NLL                       : "
    f"{selected_result['test_nll']:.8e}"
)

print(
    f"Held-out drift MSE                      : "
    f"{selected_result['drift_mse']:.8e}"
)

print(
    f"Held-out diffusion MSE                  : "
    f"{selected_result['sigma_mse']:.8e}"
)

if best_refined_candidate is not None:
    best_candidate_result = (
        branch_results[
            best_refined_candidate
        ]
    )

    candidate_validation_gain = (
        baseline_valid_nll
        - best_candidate_result[
            "valid_nll"
        ]
    )

    print(
        f"Best finite Stage-III candidate         : "
        f"{best_refined_candidate}"
    )

    print(
        f"Candidate validation NLL                : "
        f"{best_candidate_result['valid_nll']:.8e}"
    )

    print(
        f"Candidate validation gain vs baseline   : "
        f"{candidate_validation_gain:.8e}"
    )

# -----------------------------------------------------------------------------
# Save current-dimension results
# -----------------------------------------------------------------------------

results_dir = Path(
    CONFIG["E8_RESULTS_DIR"]
)

results_dir.mkdir(
    parents=True,
    exist_ok=True,
)

run_stem = (
    f"e8_{CONFIG['TRUE_SDE_NAME']}"
    f"_d{D:03d}"
    f"_seed{CONFIG['SEED']:04d}"
)

json_path = (
    results_dir
    / f"{run_stem}.json"
)

mc_csv_path = (
    results_dir
    / f"{run_stem}_mc_variance.csv"
)

def _json_safe(value):
    if isinstance(value, dict):
        return {
            key: _json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            _json_safe(item)
            for item in value
        ]

    if isinstance(
        value,
        (np.floating, float),
    ):
        value = float(value)

        if np.isfinite(value):
            return value

        return None

    if isinstance(
        value,
        (np.integer, int),
    ):
        return int(value)

    if isinstance(
        value,
        (np.bool_, bool),
    ):
        return bool(value)

    return value

serializable_result = {
    "schema_version": 2,
    "experiment": (
        "E8_dimension_and_"
        "residual_variance_scaling"
    ),
    "state_dimension": int(D),
    "coordinate_dimension": int(Q),
    "seed": int(
        CONFIG["SEED"]
    ),
    "true_sde_name": (
        CONFIG["TRUE_SDE_NAME"]
    ),
    "n_generators": int(
        CONFIG["N_GENERATORS"]
    ),
    "n_basis": int(N_BASIS),
    "evaluation_collocation_mode": (
        CONFIG[
            "E8_EVAL_COLLOCATION_MODE"
        ]
    ),
    "variance_collocation_mode": (
        CONFIG[
            "E8_VARIANCE_COLLOCATION_MODE"
        ]
    ),
    "residual_diagnostic_split": (
        CONFIG[
            "E8_RESIDUAL_DIAGNOSTIC_SPLIT"
        ]
    ),
    "mc_batch_sizes": [
        int(value)
        for value in mc_batch_sizes
    ],
    "mc_repeats": int(
        mc_repeats
    ),
    "mc_reference_branch": (
        mc_reference_branch
    ),
    "best_refined_candidate": (
        best_refined_candidate
    ),
    "selected_branch": (
        selected_branch
    ),
    "selection_reason": (
        selection_reason
    ),
    "branches": (
        branch_results
    ),
    "gradient_diagnostics_initial": (
        gradient_results_initial
    ),
    "gradient_diagnostics_final": (
        gradient_results_final
    ),
    "residual_consistency_check": (
        residual_consistency_check
    ),
    "config_snapshot": {
        key_name: value
        for key_name, value
        in CONFIG.items()
        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
                list,
                tuple,
            ),
        )
    },
}

with open(
    json_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        _json_safe(
            serializable_result
        ),
        file,
        indent=2,
    )

with open(
    mc_csv_path,
    "w",
    newline="",
    encoding="utf-8",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(
            mc_rows[0].keys()
        ),
    )

    writer.writeheader()
    writer.writerows(
        mc_rows
    )

print(
    "\nSaved current-dimension files:"
)

print(
    " ",
    json_path,
)

print(
    " ",
    mc_csv_path,
)

# -----------------------------------------------------------------------------
# Aggregate completed dimensions
# -----------------------------------------------------------------------------

aggregate_by_dimension = {}

for candidate_path in sorted(
    results_dir.glob(
        "e8_*.json"
    )
):
    try:
        with open(
            candidate_path,
            "r",
            encoding="utf-8",
        ) as file:
            candidate = json.load(
                file
            )

    except Exception:
        continue

    if (
        candidate.get(
            "experiment"
        )
        != (
            "E8_dimension_and_"
            "residual_variance_scaling"
        )
    ):
        continue

    if (
        candidate.get(
            "true_sde_name"
        )
        != CONFIG[
            "TRUE_SDE_NAME"
        ]
    ):
        continue

    if (
        candidate.get(
            "seed"
        )
        != CONFIG["SEED"]
    ):
        continue

    dimension = int(
        candidate[
            "state_dimension"
        ]
    )

    base_result = (
        candidate[
            "branches"
        ]["baseline"]
    )

    selected_name = candidate.get(
        "selected_branch",
        candidate.get(
            "best_normalized_branch"
        ),
    )

    if selected_name is not None:
        selected_dimension_result = (
            candidate[
                "branches"
            ].get(
                selected_name
            )
        )
    else:
        selected_dimension_result = None

    initial_gradient_dictionary = (
        candidate.get(
            "gradient_diagnostics_initial",
            candidate.get(
                "gradient_diagnostics",
                {},
            ),
        )
    )

    if (
        "baseline"
        not in initial_gradient_dictionary
    ):
        continue

    baseline_gradient_result = (
        initial_gradient_dictionary[
            "baseline"
        ]
    )

    aggregate_by_dimension[
        dimension
    ] = {
        "dimension": dimension,

        "drift_raw": (
            base_result[
                "residual_drift_raw"
            ]
        ),

        "drift_per_component": (
            base_result[
                "residual_drift_per_component"
            ]
        ),

        "diffusion_raw": (
            base_result[
                "residual_diffusion_raw"
            ]
        ),

        "diffusion_per_full_entry": (
            base_result[
                "residual_diffusion_per_full_entry"
            ]
        ),

        "normalized_total": (
            base_result[
                "residual_normalized_total"
            ]
        ),

        "grad_data": (
            baseline_gradient_result[
                "grad_data_norm"
            ]
        ),

        "grad_sym_normalized": (
            baseline_gradient_result[
                "grad_normalized_symmetry_norm"
            ]
        ),

        "grad_sym_to_data": (
            baseline_gradient_result[
                "unweighted_sym_to_data_ratio"
            ]
        ),

        "selected_branch": (
            selected_name
        ),

        "selected_lambda": (
            selected_dimension_result[
                "lambda_sym"
            ]
            if (
                selected_dimension_result
                is not None
            )
            else np.nan
        ),

        "selected_drift_mse": (
            selected_dimension_result[
                "drift_mse"
            ]
            if (
                selected_dimension_result
                is not None
            )
            else np.nan
        ),

        "selected_sigma_mse": (
            selected_dimension_result[
                "sigma_mse"
            ]
            if (
                selected_dimension_result
                is not None
            )
            else np.nan
        ),

        "selected_valid_nll": (
            selected_dimension_result[
                "valid_nll"
            ]
            if (
                selected_dimension_result
                is not None
            )
            else np.nan
        ),

        "selected_test_nll": (
            selected_dimension_result[
                "test_nll"
            ]
            if (
                selected_dimension_result
                is not None
            )
            else np.nan
        ),

        "source_file": (
            candidate_path.name
        ),
    }

aggregate_rows = [
    aggregate_by_dimension[
        dimension
    ]
    for dimension in sorted(
        aggregate_by_dimension
    )
]

required_dimensions = set(
    CONFIG["E8_DIMENSIONS"]
)

available_dimensions = {
    row["dimension"]
    for row in aggregate_rows
}

missing_dimensions = sorted(
    required_dimensions
    - available_dimensions
)

print(
    "\n"
    + "=" * 108
)

print(
    "E8 CROSS-DIMENSION AGGREGATE"
)

print(
    "=" * 108
)

if aggregate_rows:
    header = (
        f"{'d':>5s} "
        f"{'Rf raw':>13s} "
        f"{'Rf/d':>13s} "
        f"{'RS raw':>13s} "
        f"{'RS/d^2':>13s} "
        f"{'Lnorm':>13s} "
        f"{'|g_sym|/|g_data|':>18s} "
        f"{'lambda*':>12s}"
    )

    print(
        header
    )

    print(
        "-" * len(header)
    )

    for row in aggregate_rows:
        print(
            f"{row['dimension']:5d} "
            f"{row['drift_raw']:13.5e} "
            f"{row['drift_per_component']:13.5e} "
            f"{row['diffusion_raw']:13.5e} "
            f"{row['diffusion_per_full_entry']:13.5e} "
            f"{row['normalized_total']:13.5e} "
            f"{row['grad_sym_to_data']:18.5e} "
            f"{row['selected_lambda']:12.5e}"
        )

else:
    print(
        "No compatible completed runs found."
    )

if missing_dimensions:
    print(
        "\nMissing dimensions:",
        missing_dimensions,
    )

    print(
        "Restart the runtime and run the "
        "notebook once for each missing "
        "STATE_DIM."
    )

else:
    dimensions_array = np.asarray(
        [
            row["dimension"]
            for row in aggregate_rows
        ],
        dtype=float,
    )

    def fit_log_slope(field_name):
        values = np.asarray(
            [
                row[field_name]
                for row in aggregate_rows
            ],
            dtype=float,
        )

        valid = (
            np.isfinite(values)
            & (values > 0.0)
            & np.isfinite(
                dimensions_array
            )
            & (
                dimensions_array > 0.0
            )
        )

        if np.sum(valid) < 2:
            return (
                np.nan,
                np.nan,
            )

        slope, intercept = np.polyfit(
            np.log(
                dimensions_array[
                    valid
                ]
            ),
            np.log(
                values[
                    valid
                ]
            ),
            1,
        )

        predicted = (
            slope
            * np.log(
                dimensions_array[
                    valid
                ]
            )
            + intercept
        )

        observed = np.log(
            values[
                valid
            ]
        )

        residual_ss = np.sum(
            (
                observed
                - predicted
            ) ** 2
        )

        total_ss = np.sum(
            (
                observed
                - np.mean(
                    observed
                )
            ) ** 2
        )

        r_squared = (
            1.0
            - residual_ss
            / (
                total_ss
                + 1e-12
            )
        )

        return (
            float(slope),
            float(r_squared),
        )

    print(
        "\nEmpirical power laws "
        "y approximately proportional "
        "to d^alpha"
    )

    scaling_fields = [
        (
            "drift_raw",
            "raw drift residual",
        ),
        (
            "drift_per_component",
            "drift residual per component",
        ),
        (
            "diffusion_raw",
            "raw diffusion residual",
        ),
        (
            "diffusion_per_full_entry",
            "diffusion residual per full entry",
        ),
        (
            "normalized_total",
            "normalized total residual",
        ),
        (
            "grad_sym_to_data",
            "normalized symmetry/data "
            "gradient ratio",
        ),
    ]

    for (
        field_name,
        field_label,
    ) in scaling_fields:
        alpha, r_squared = (
            fit_log_slope(
                field_name
            )
        )

        print(
            f"  {field_label:<48s} "
            f"alpha={alpha: .5f}, "
            f"log-log R^2="
            f"{r_squared: .5f}"
        )

    selected_lambdas = np.asarray(
        [
            row[
                "selected_lambda"
            ]
            for row in aggregate_rows
        ],
        dtype=float,
    )

    finite_selected_lambdas = (
        selected_lambdas[
            np.isfinite(
                selected_lambdas
            )
            & (
                selected_lambdas
                > 0.0
            )
        ]
    )

    if (
        finite_selected_lambdas.size
    ):
        selected_lambda_factor = (
            np.max(
                finite_selected_lambdas
            )
            / np.min(
                finite_selected_lambdas
            )
        )

        print(
            "\nSelected nonzero-lambda "
            "consistency"
        )

        print(
            f"  minimum selected lambda               : "
            f"{np.min(finite_selected_lambdas):.8e}"
        )

        print(
            f"  maximum selected lambda               : "
            f"{np.max(finite_selected_lambdas):.8e}"
        )

        print(
            f"  max/min factor                        : "
            f"{selected_lambda_factor:.6f}"
        )

    else:
        print(
            "\nNo dimension selected a "
            "nonzero Stage-III lambda."
        )

aggregate_csv_path = (
    results_dir
    / (
        f"e8_{CONFIG['TRUE_SDE_NAME']}"
        f"_seed{CONFIG['SEED']:04d}"
        f"_cross_dimension_summary.csv"
    )
)

if aggregate_rows:
    with open(
        aggregate_csv_path,
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=list(
                aggregate_rows[
                    0
                ].keys()
            ),
        )

        writer.writeheader()

        writer.writerows(
            aggregate_rows
        )

    print(
        "\nSaved aggregate CSV:",
        aggregate_csv_path,
    )

print(
    "\n"
    + "=" * 108
)

print(
    "END OF E8 REPORT"
)

print(
    "=" * 108
)